<div style="width: 76ch;">
<h3>Vierstangen-mechanisme: Python code</h3>
(<em>F. Ulloa Roas, H. Bruyninckx</em>)

<div style="height: 15rem;">
 <img style="border: 0; display: inline-block;" height="100%"
      src="images/planar-four-bar-linkage-PQRS.svg"
 >
</div>

</div>

In [ ]:
# Animation of four bar

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib widget

In [ ]:
# kinematic parameters, in SI units:
r1   = 1.40 #m
r2   = 0.50 #m
r3   = 1.52 #m
r4   = 0.70 #m
phi1 = 0    #rad

# configuration of numerical simulation interval and sampling:
t_begin =  0     # start time of simulation
t_end   = 25     # end time of simulation
Ts      =  0.10  # time step of simulation
t = np.arange(t_begin, t_end + Ts, Ts)  # time vector

# angular position driver:
omega  = 0.5      # driver frequency [rad/s]
A      = 1        # amplitude []
phi2   = 1 + A * np.sin(omega * t)
dphi2  = omega * A * np.cos(omega * t)
ddphi2 = -omega ** 2 * A * np.sin(omega * t)

# initial conditions for nonlinear solver ("fsolve") of position closure:
phi3_init = 0.
phi4_init = 0.2 
# (different choices can lead to different topologies of your mechanism)


In [ ]:
# Function to rotate a vector z over an angle theta:
def rotate_vector(z, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
    return np.dot(rotation_matrix, z)

In [ ]:
# Define function to compute "gap" in position closure:
def loop_closure_eqs(phi_init, phi2, r1, r2, r3, r4, phi1):
    phi3 = phi_init[0]
    phi4 = phi_init[1]

    # Loop closure gaps:
    F1 = r2 * np.cos(phi2) + r3 * np.cos(phi3) - r4 * np.cos(phi4) - r1 * np.cos(phi1)
    F2 = r2 * np.sin(phi2) + r3 * np.sin(phi3) - r4 * np.sin(phi4) - r1 * np.sin(phi1)

    return [F1, F2]

In [ ]:
# Function to compute the kinematics of four-bar mechanism:

def kinematics_4bar(r1, r2, r3, r4, phi1, phi2, dphi2, ddphi2, phi3_init, phi4_init, t):
  optim_options = {"full_output":True}  # options for fsolve

  # numerically solving the position closure at all sampling times:
  for k, time in enumerate(t):
      # Position Analysis
      x, _, ier, message  = fsolve(lambda x: loop_closure_eqs(x, phi2[k], r1, r2, r3, r4, phi1), [phi3_init, phi4_init], **optim_options)

      if ier != 1:
          print("The fsolve exit flag was not 1, probably no convergence!")
          print(message)

      phi3[k] = x[0]
      phi4[k] = x[1]

      # velocity closure:
      A = np.array([[-r3 * np.sin(phi3[k]), r4 * np.sin(phi4[k])],
                    [r3 * np.cos(phi3[k]), -r4 * np.cos(phi4[k])]])
      B = np.array([r2 * np.sin(phi2[k]) * dphi2[k],
                    -r2 * np.cos(phi2[k]) * dphi2[k]])

      x = np.linalg.solve(A, B)
      dphi3[k] = x[0]
      dphi4[k] = x[1]
      cond[k]  = np.linalg.cond(A)

      # acceleration closure:
      B = np.array([r2 * np.cos(phi2[k]) * dphi2[k]**2 + r2 * np.sin(phi2[k]) * ddphi2[k] + r3 * np.cos(phi3[k]) * dphi3[k]**2 - r4 * np.cos(phi4[k]) * dphi4[k]**2,
                    r2 * np.sin(phi2[k]) * dphi2[k]**2 - r2 * np.cos(phi2[k]) * ddphi2[k] + r3 * np.sin(phi3[k]) * dphi3[k]**2 - r4 * np.sin(phi4[k]) * dphi4[k]**2])

      # Note: A matrix is the same as for velocities.
      x = np.linalg.solve(A, B)
      ddphi3[k] = x[0]
      ddphi4[k] = x[1]

      # Next iteration with initial values from by simple integration:
      phi3_init = phi3[k] + (t[1] - t[0]) * dphi3[k]
      phi4_init = phi4[k] + (t[1] - t[0]) * dphi4[k]

  return phi3, phi4, dphi3, dphi4, ddphi3, ddphi4, cond

In [ ]:
# compute the kinematics at all sample times "t":

t_size = len(t)              # number of iterations
sim_fraction = 2             # fraction of simulated frames in animation
frames = t_size/sim_fraction # number of animated iterations
delta = int(np.floor(t_size/frames))    # time between frames
index_vec = np.arange(0, t_size, delta) # index of animated frames        

phi3   = np.zeros_like(t)
phi4   = np.zeros_like(t)
dphi3  = np.zeros_like(t)
dphi4  = np.zeros_like(t)
ddphi3 = np.zeros_like(t)
ddphi4 = np.zeros_like(t)

cond   = np.zeros_like(t) # condition number of matrix A

[phi3, phi4, dphi3, dphi4, ddphi3, ddphi4, cond] = kinematics_4bar(r1, r2, r3, r4, phi1, phi2, dphi2, ddphi2, phi3_init, phi4_init, t)


In [ ]:
# plot position, velocity and acceleration profiles:

fig1, ax1 = plt.subplots(nrows=3,ncols=3,constrained_layout=True)
fig1.suptitle("Position, velocity and acceleration")

ax1[0,0].plot(t, phi2)
ax1[0,0].set_ylabel(r'$\phi_2$ [rad]')

ax1[0,1].plot(t, phi3)
ax1[0,1].set_ylabel(r'$\phi_3$ [rad]')

ax1[0,2].plot(t, phi4)
ax1[0,2].set_ylabel(r'$\phi_4$ [rad]')

ax1[1,0].plot(t, dphi2)
ax1[1,0].set_ylabel(r'$\dot{\phi}_2$ [rad/s]')

ax1[1,1].plot(t, dphi3)
ax1[1,1].set_ylabel(r'$\dot{\phi}_3$ [rad/s]')

ax1[1,2].plot(t, dphi4)
ax1[1,2].set_ylabel(r'$\dot{\phi}_4$ [rad/s]')

ax1[2,0].plot(t, ddphi2)
ax1[2,0].set_ylabel(r'$\ddot{\phi}_2 [\text{rad/s}^2]$')
ax1[2,0].set_xlabel('t [s]')

ax1[2,1].plot(t, ddphi3)
ax1[2,1].set_ylabel(r'$\ddot{\phi}_3 [\text{rad/s}^2]$')
ax1[2,1].set_xlabel('t [s]')

ax1[2,2].plot(t, ddphi4)
ax1[2,2].set_ylabel(r'$\ddot{\phi}_4 [\text{rad/s}^2]$')
ax1[2,2].set_xlabel('t [s]')

plt.show()

# plot of condition number:
figc, axc = plt.subplots()
figc.suptitle("Condition number of matrix A")

axc.plot(t, cond)
axc.set_ylabel("cond(A)")
plt.show()


In [ ]:
# Animation of mechanism motion
plt.ioff()               # don't show figure until drawn explicitly:
fig2, ax = plt.subplots() # Initialize the figure

# Define point P and S as ground link:
P = np.array([0,0])
S = rotate_vector(np.array([r1, 0]), phi1)

# compute width and height of drawing canvas:
x_left   = -1.25 * r2
y_bottom = -1.25 * max(r2, r4)
x_right  = r1 + 1.25 * r4
y_top    = 1.25 * max(r2, r4)

# function to update the animation every "delta" steps:
def update(frame):
    global fig2, ax, x_left, x_right, y_bottom, y_top, P, S, index_vec
    ax.cla()         # Clear the current axes
    ax.axis('equal') # Ensures 1 unit on x == 1 unit on y
    ax.set_xlabel('[m]')
    ax.set_ylabel('[m]')
    ax.set_xlim([x_left, x_right])
    time_now = t[frame]
    ax.set_title(f'Time {time_now: .2f}')
    #ax.set_title(f'Frame {frame}')

    # Calculate Cartesian coordinates of mechanism joints:
    Q = P + rotate_vector(np.array([r2, 0]), phi2[frame])
    R1 = Q + rotate_vector(np.array([r3, 0]), phi3[frame])
    R2 = S + rotate_vector(np.array([r4, 0]), phi4[frame])
    loop1 = np.array([P, Q, R1, R2, S])
    ax.plot(loop1[:, 0], loop1[:, 1], '-o')
    return(ax)

# animate the mechanism:
ani = FuncAnimation(fig2, update, frames=len(index_vec), interval=50, repeat='false')


In [ ]:
# display mechanism animation:
ani_html = ani.to_jshtml(default_mode='once')
HTML(ani_html)

In [ ]:
# dynamic parameters, defined in a local frame on each of the bars:
X2 = r2 / 2    # X coordinates of cog (centre of gravity)
X3 = r3 / 2
X4 = r4 / 2

Y2 = 0         # Y coordinates of cog
Y3 = 0  
Y4 = 0

m2 = r2 * 15.4 # mass of round steel bar of 5cm diameter
m3 = r3 * 15.4
m4 = r4 * 15.4

# moments of inertia at center of mass (formula of a thin rod):
J2 = m2 * r2 ** 2 / 12
J3 = m3 * r3 ** 2 / 12
J4 = m4 * r4 ** 2 / 12


In [ ]:
# formulas for inverse dynamics:
def dynamics_4bar ( phi2, phi3, phi4, dphi2, dphi3, dphi4, ddphi2, ddphi3, ddphi4, r2, r3, r4, m2, m3, m4, X2, X3, X4, Y2, Y3, Y4, J2, J3, J4, t ):
    # Define vectors from the center of gravity of each bar to point P, Q, R, S
    cog2_P_x = -X2 * np.cos(phi2) - Y2 * np.cos(phi2 + np.pi / 2)
    cog2_P_y = -X2 * np.sin(phi2) - Y2 * np.sin(phi2 + np.pi / 2)
    cog2_Q_x = (r2 - X2) * np.cos(phi2) - Y2 * np.cos(phi2 + np.pi / 2)
    cog2_Q_y = (r2 - X2) * np.sin(phi2) - Y2 * np.sin(phi2 + np.pi / 2)
    cog3_Q_x = -Y3 * np.cos(phi3 + np.pi / 2) - X3 * np.cos(phi3)
    cog3_Q_y = -Y3 * np.sin(phi3 + np.pi / 2) - X3 * np.sin(phi3)
    cog3_R_x = (r3 - X3) * np.cos(phi3) - Y3 * np.cos(phi3 + np.pi / 2)
    cog3_R_y = (r3 - X3) * np.sin(phi3) - Y3 * np.sin(phi3 + np.pi / 2)
    cog4_S_x = -X4 * np.cos(phi4) - Y4 * np.cos(phi4 + np.pi / 2)
    cog4_S_y = -X4 * np.sin(phi4) - Y4 * np.sin(phi4 + np.pi / 2)
    cog4_R_x = (r4 - X4) * np.cos(phi4) - Y4 * np.cos(phi4 + np.pi / 2)
    cog4_R_y = (r4 - X4) * np.sin(phi4) - Y4 * np.sin(phi4 + np.pi / 2)

    # Define 3D vectors omega (dphi) and alpha (ddphi)
    omega2 = np.column_stack((np.zeros_like(phi2), np.zeros_like(phi2), dphi2))
    omega3 = np.column_stack((np.zeros_like(phi2), np.zeros_like(phi2), dphi3))
    omega4 = np.column_stack((np.zeros_like(phi2), np.zeros_like(phi2), dphi4))
    alpha2 = np.column_stack((np.zeros_like(phi2), np.zeros_like(phi2), ddphi2))
    alpha3 = np.column_stack((np.zeros_like(phi2), np.zeros_like(phi2), ddphi3))
    alpha4 = np.column_stack((np.zeros_like(phi2), np.zeros_like(phi2), ddphi4))

    # Define 3D model vectors
    P_cog2_vec = np.column_stack((-cog2_P_x, -cog2_P_y, np.zeros_like(phi2)))
    Q_cog3_vec = np.column_stack((-cog3_Q_x, -cog3_Q_y, np.zeros_like(phi2)))
    S_cog4_vec = np.column_stack((-cog4_S_x, -cog4_S_y, np.zeros_like(phi2)))
    PQ_vec = np.column_stack((r2 * np.cos(phi2), r2 * np.sin(phi2), np.zeros_like(phi2)))

    # Define acceleration vectors
    acc_2 = np.cross(omega2, np.cross(omega2, P_cog2_vec)) + np.cross(alpha2, P_cog2_vec)
    acc_Q = np.cross(omega2, np.cross(omega2, PQ_vec)) + np.cross(alpha2, PQ_vec)
    acc_3 = acc_Q + np.cross(omega3, np.cross(omega3, Q_cog3_vec)) + np.cross(alpha3, Q_cog3_vec)
    acc_4 = np.cross(omega4, np.cross(omega4, S_cog4_vec)) + np.cross(alpha4, S_cog4_vec)
    acc_2x = acc_2[:, 0]
    acc_2y = acc_2[:, 1]
    acc_3x = acc_3[:, 0]
    acc_3y = acc_3[:, 1]
    acc_4x = acc_4[:, 0]
    acc_4y = acc_4[:, 1]

    # Calculate dynamics for each time step
    t_size = len(t)
    for k in range(t_size):
        A = np.array([[1, 0, 1, 0, 0, 0, 0, 0, 0],
                      [0, 1, 0, 1, 0, 0, 0, 0, 0],
                      [0, 0, -1, 0, -1, 0, 0, 0, 0],
                      [0, 0, 0, -1, 0, -1, 0, 0, 0],
                      [0, 0, 0, 0, 1, 0, 1, 0, 0],
                      [0, 0, 0, 0, 0, 1, 0, 1, 0],
                      [-cog2_P_y[k], cog2_P_x[k], -cog2_Q_y[k], cog2_Q_x[k], 0, 0, 0, 0, 1],
                      [0, 0, cog3_Q_y[k], -cog3_Q_x[k], cog3_R_y[k], -cog3_R_x[k], 0, 0, 0],
                      [0, 0, 0, 0, -cog4_R_y[k], cog4_R_x[k], -cog4_S_y[k], cog4_S_x[k], 0]])

        B = np.array([m2 * acc_2x[k],
                      m2 * acc_2y[k],
                      m3 * acc_3x[k],
                      m3 * acc_3y[k],
                      m4 * acc_4x[k],
                      m4 * acc_4y[k],
                      J2 * ddphi2[k],
                      J3 * ddphi3[k],
                      J4 * ddphi4[k]])

        x = np.linalg.lstsq(A, B, rcond=None)[0]

      # Save results
        F_P_x[k] = x[0]
        F_P_y[k] = x[1]
        F_Q_x[k] = x[2]
        F_Q_y[k] = x[3]
        F_R_x[k] = x[4]
        F_R_y[k] = x[5]
        F_S_x[k] = x[6]
        F_S_y[k] = x[7]
        M_P[k] = x[8]

    return F_P_x, F_Q_x, F_R_x, F_S_x, F_P_y, F_Q_y, F_R_y, F_S_y, M_P

# compute inverse dynamics:
# Allocate matrices for force (F) and moment (M)
F_P_x = np.zeros_like(t)
F_P_y = np.zeros_like(t)
F_Q_x = np.zeros_like(t)
F_Q_y = np.zeros_like(t)
F_R_x = np.zeros_like(t)
F_R_y = np.zeros_like(t)
F_S_x = np.zeros_like(t)
F_S_y = np.zeros_like(t)
M_P   = np.zeros_like(t)

[F_P_x, F_Q_x, F_R_x, F_S_x, F_P_y, F_Q_y, F_R_y, F_S_y, M_P] = dynamics_4bar(phi2, phi3, phi4, dphi2, dphi3, dphi4, ddphi2, ddphi3, ddphi4, r2, r3, r4, m2, m3, m4, X2, X3, X4, Y2, Y3, Y4, J2, J3, J4, t )


In [ ]:
# plot forces and moments:
fig2, ax2 = plt.subplots(nrows=2,ncols=2,constrained_layout=True, figsize=(10,5))
fig2.suptitle("Forces")

ax2[0,0].plot(t, F_P_x, label="x")
ax2[0,0].plot(t, F_P_y, label="y")
ax2[0,0].set_ylabel(r'$F_{Px}, F_{Py}$ [N]')
ax2[0,0].set_xlabel('t [s]')
ax2[0,0].legend()

ax2[0,1].plot(t, F_Q_x)
ax2[0,1].plot(t, F_Q_y)
ax2[0,1].set_ylabel(r'$F_{Qx}, F_{Qy}$ [N]')
ax2[0,1].set_xlabel('t [s]')

ax2[1,0].plot(t, F_R_x)
ax2[1,0].plot(t, F_R_y)
ax2[1,0].set_ylabel(r'$F_{Rx}, F_{Ry}$ [N]')
ax2[1,0].set_xlabel('t [s]')

ax2[1,1].plot(t, F_S_x)
ax2[1,1].plot(t, F_S_y)
ax2[1,1].set_ylabel(r'$F_{Sx}, F_{Sy}$ [N]')
ax2[1,1].set_xlabel('t [s]')

plt.show()

fig3, ax3 = plt.subplots(constrained_layout=True, figsize=(5,2.5))
fig3.suptitle("Moment $M_P$")

ax3.plot(t, M_P)
ax3.set_ylabel(r'$M_P$ [Nm]')
ax3.set_xlabel('t [s]')

plt.show()
